In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
possible_paths = [
    Path("../data/processed/olist_orders_abt.csv"),
    Path("data/processed/olist_orders_abt.csv"),
    Path("olist_orders_abt.csv")
]

data_path = None

for path in possible_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find olist_orders_abt.csv. Please check the file path."
    )

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("File:", data_path)
print("Shape:", df.shape)

Dataset loaded successfully!
File: ..\data\processed\olist_orders_abt.csv
Shape: (99441, 29)


In [3]:
print("First 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)

First 5 rows:


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,...,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017,10,2,0,...,38.71,1.0,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018,7,24,1,...,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018,8,8,2,...,179.12,3.0,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,delivered,2017,11,18,5,...,72.20,1.0,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018,2,13,1,...,28.62,1.0,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,stationery



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  99441 non-null  str    
 1   customer_id               99441 non-null  str    
 2   customer_unique_id        99441 non-null  str    
 3   customer_city             99441 non-null  str    
 4   customer_state            99441 non-null  str    
 5   order_status              99441 non-null  str    
 6   order_year                99441 non-null  int64  
 7   order_month               99441 non-null  int64  
 8   order_day                 99441 non-null  int64  
 9   order_day_of_week         99441 non-null  int64  
 10  order_hour                99441 non-null  int64  
 11  delivery_days             96476 non-null  float64
 12  estimated_delivery_days   99441 non-null  float64
 13  delivery_delay_days       96476 non-null  float64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_city,99441,4119,sao paulo,15540,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,99441,27,SP,41746,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,99441,8,delivered,96478,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_year,99441.0,NaN,NaN,NaN,2017.539838,0.505007,2016.0,2017.0,2018.0,2018.0,2018.0
order_month,99441.0,NaN,NaN,NaN,6.03222,3.232999,1.0,3.0,6.0,8.0,12.0
order_day,99441.0,NaN,NaN,NaN,15.505948,8.667298,1.0,8.0,15.0,23.0,31.0
order_day_of_week,99441.0,NaN,NaN,NaN,2.755735,1.966495,0.0,1.0,3.0,4.0,6.0


In [4]:
target = "is_late_delivery"

if target not in df.columns:
    raise ValueError(
        f"Target column '{target}' not found in the dataset."
    )

print("Target column:", target)

print("\nTarget counts:")
print(df[target].value_counts())

print("\nTarget percentage:")
print(df[target].value_counts(normalize=True) * 100)

Target column: is_late_delivery

Target counts:
is_late_delivery
0    91614
1     7827
Name: count, dtype: int64

Target percentage:
is_late_delivery
0    92.129001
1     7.870999
Name: proportion, dtype: float64


In [5]:
identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

columns_to_remove = identifier_columns + leakage_columns + [target]

columns_to_remove = [
    col for col in columns_to_remove
    if col in df.columns
]

print("Columns to remove:")
print(columns_to_remove)

Columns to remove:
['order_id', 'customer_id', 'customer_unique_id', 'delivery_days', 'delivery_delay_days', 'review_score', 'review_comment_count', 'has_review_comment', 'is_low_review', 'is_late_delivery']


In [6]:
X = df.drop(columns=columns_to_remove)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

X shape: (99441, 19)
y shape: (99441,)

X columns:
['customer_city', 'customer_state', 'order_status', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'main_product_category']


In [8]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers']

Categorical features:
['customer_city', 'customer_state', 'order_status', 'dominant_payment_type', 'main_product_category']


In [9]:
print("Missing numerical values:")
display(
    X[numeric_features]
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nMissing categorical values:")
display(
    X[categorical_features]
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Missing numerical values:


unique_products             775
unique_sellers              775
total_price                 775
total_freight               775
total_items                 775
payment_types_count           1
total_payment_value           1
max_payment_installments      1
estimated_delivery_days       0
order_hour                    0
order_month                   0
order_year                    0
order_day_of_week             0
order_day                     0
dtype: int64


Missing categorical values:


main_product_category    2185
dominant_payment_type       1
customer_city               0
order_status                0
customer_state              0
dtype: int64

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (79552, 19)
X_test shape: (19889, 19)
y_train shape: (79552,)
y_test shape: (19889,)


In [11]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Numerical pipeline created!")

Numerical pipeline created!


In [12]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

print("Categorical pipeline created!")

Categorical pipeline created!


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("ColumnTransformer created!")

ColumnTransformer created!


In [14]:
model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

print("Complete ML pipeline created!")

Complete ML pipeline created!


In [15]:
model_pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Pipeline trained successfully!


In [16]:
y_pred = model_pipeline.predict(X_test)

print("First 20 predictions:")
print(y_pred[:20])

First 20 predictions:
[0 1 0 0 0 0 0 0 0 0 0 1 1 1 0 0 1 1 0 0]


In [17]:
y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("First 10 predicted probabilities:")
print(y_pred_proba[:10])

First 10 predicted probabilities:
[0.28209105 0.63601995 0.0513133  0.1066715  0.49171384 0.43043674
 0.2669385  0.49776722 0.44104213 0.48549816]


In [18]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.6591080496757001


In [19]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[12132  6192]
 [  588   977]]


In [20]:
print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.66      0.78     18324
           1       0.14      0.62      0.22      1565

    accuracy                           0.66     19889
   macro avg       0.55      0.64      0.50     19889
weighted avg       0.89      0.66      0.74     19889



In [21]:
report = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

print("Accuracy:", accuracy)
print("Precision:", report["1"]["precision"])
print("Recall:", report["1"]["recall"])
print("F1-score:", report["1"]["f1-score"])

Accuracy: 0.6591080496757001
Precision: 0.13628121076858696
Recall: 0.6242811501597444
F1-score: 0.22372337989466454


In [22]:
feature_names = (
    model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("First 20 transformed feature names:")
print(feature_names[:20])

print("\nTotal transformed features:", len(feature_names))

First 20 transformed feature names:
['num__order_year' 'num__order_month' 'num__order_day'
 'num__order_day_of_week' 'num__order_hour' 'num__estimated_delivery_days'
 'num__total_payment_value' 'num__max_payment_installments'
 'num__payment_types_count' 'num__total_items' 'num__total_price'
 'num__total_freight' 'num__unique_products' 'num__unique_sellers'
 'cat__customer_city_abadia dos dourados' 'cat__customer_city_abadiania'
 'cat__customer_city_abaete' 'cat__customer_city_abaetetuba'
 'cat__customer_city_abaira' 'cat__customer_city_abare']

Total transformed features: 3998


In [23]:
feature_names_path = Path("models")

feature_names_path.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    feature_names_path / "feature_names.npy",
    feature_names
)

print("Feature names saved!")

Feature names saved!


In [24]:
model_path = Path("models/late_delivery_pipeline.joblib")

model_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    model_pipeline,
    model_path
)

print("Pipeline saved at:")
print(model_path)

Pipeline saved at:
models\late_delivery_pipeline.joblib


In [25]:
loaded_pipeline = joblib.load(
    model_path
)

print("Pipeline loaded successfully!")

Pipeline loaded successfully!


In [26]:
loaded_predictions = loaded_pipeline.predict(X_test)

print("First 10 loaded predictions:")
print(loaded_predictions[:10])

First 10 loaded predictions:
[0 1 0 0 0 0 0 0 0 0]


In [27]:
print(
    "Predictions match:",
    np.array_equal(y_pred, loaded_predictions)
)

Predictions match: True


In [28]:
sample_order = X_test.iloc[[0]]

print("Sample order:")
display(sample_order)

Sample order:


,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,order_hour,estimated_delivery_days,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
10248,mage,RJ,delivered,2018,2,25,6,19,24.18265,127.92,6.0,1.0,credit_card,1.0,109.9,18.02,1.0,1.0,bed_bath_table


In [29]:
sample_prediction = loaded_pipeline.predict(
    sample_order
)

print("Predicted class:", sample_prediction[0])

Predicted class: 0


In [30]:
sample_probability = loaded_pipeline.predict_proba(
    sample_order
)[:, 1]

print(
    "Probability of late delivery:",
    sample_probability[0]
)

Probability of late delivery: 0.2820910538024383


In [31]:
if sample_probability[0] >= 0.5:
    print("Prediction: Likely late delivery")
else:
    print("Prediction: Likely not late delivery")

Prediction: Likely not late delivery


In [32]:
print("========== LAB 4 FINAL CHECK ==========")

print("Dataset shape:", df.shape)
print("X shape:", X.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nNumber of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nAccuracy:", accuracy)
print("Precision:", report["1"]["precision"])
print("Recall:", report["1"]["recall"])
print("F1-score:", report["1"]["f1-score"])

print("\nTransformed features:", len(feature_names))

print("\nPipeline file:", model_path)
print("\nLAB 4 COMPLETED SUCCESSFULLY!")

========== LAB 4 FINAL CHECK ==========
Dataset shape: (99441, 29)
X shape: (99441, 19)
X_train shape: (79552, 19)
X_test shape: (19889, 19)

Number of numerical features: 14
Number of categorical features: 5

Accuracy: 0.6591080496757001
Precision: 0.13628121076858696
Recall: 0.6242811501597444
F1-score: 0.22372337989466454

Transformed features: 3998

Pipeline file: models\late_delivery_pipeline.joblib

LAB 4 COMPLETED SUCCESSFULLY!


In [ ]:
## Reflection

### 1. What does accuracy measure?

Accuracy measures the proportion of total predictions that were correct.

### 2. Why can accuracy be misleading?

Accuracy can be misleading when the target classes are imbalanced because a model may perform well on the majority class while performing poorly on the minority class.

### 3. What does recall measure?

Recall measures how many of the actual positive cases were correctly identified by the model.

### 4. Why may recall be important for late delivery prediction?

Recall is important because missing an actually late order can be important in a delivery prediction system.

### 5. Why does the number of features increase after preprocessing?

Categorical variables are converted into multiple binary columns using one-hot encoding.

### 6. Why is saving the complete pipeline important?

Saving the complete pipeline preserves both preprocessing and the trained model so that future data can be transformed consistently before prediction.